### Includes: vent_parameters.sql and getVentilationParams.sql

In [3]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
from datetime import timedelta
import re

# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to the MIMIC-IV dataset
mimic_path = "/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0"
output_path = '/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data'
vaso_path = '/content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/'

Mounted at /content/drive


In [ ]:
import pandas as pd

def create_vent_parameters(mimic_path, output_path):
    """
    Python conversion of ventparameters.sql for MIMIC-III
    Retrieves and aggregates ventilator parameters (PEEP, tidal volume, plateau pressure) by charttime.

    Parameters:
        mimic_path (str): Path to MIMIC-III chartevents data (.csv or .csv.gz)
        output_path (str): Path to save the output DataFrame as CSV

    Returns:
        pd.DataFrame: DataFrame of grouped ventilator parameters
    """
    print("Loading chartevents table...")
    ce = pd.read_csv(f"{mimic_path}/icu/chartevents.csv.gz",
                     usecols=['stay_id', 'subject_id', 'hadm_id', 'charttime',
                              'itemid', 'valuenum', 'value'],
                               parse_dates=['charttime'])

    # Map itemids to parameters
    PEEP_itemids = [60, 437, 505, 506, 686, 220339, 224699, 224700]
    tv_itemids = [639, 654, 681, 682, 683, 684, 224685, 224684, 224686]
    plateau_itemids = [543, 224696]

    # Filter only relevant itemids and non-null values, exclude error rows
    filt = (ce['value'].notnull()) & (
        ce['itemid'].isin(PEEP_itemids + tv_itemids + plateau_itemids)
    )
    ce_filt = ce.loc[filt, [
        'stay_id', 'subject_id', 'hadm_id', 'charttime', 'itemid', 'valuenum'
    ]].copy()

    print("Filtered Data Created")

    # Assign parameter columns based on itemid
    ce_filt['PEEP'] = ce_filt.apply(lambda x: x['valuenum'] if x['itemid'] in PEEP_itemids else None, axis=1)
    ce_filt['tidal_volume'] = ce_filt.apply(lambda x: x['valuenum'] if x['itemid'] in tv_itemids else None, axis=1)
    ce_filt['plateau_pressure'] = ce_filt.apply(lambda x: x['valuenum'] if x['itemid'] in plateau_itemids else None, axis=1)

    # Group and average, matching SQL aggregation
    aggregate = ce_filt.groupby(
        ['stay_id', 'subject_id', 'hadm_id', 'charttime'],
        as_index=False
    ).agg({
        'PEEP': 'mean',
        'tidal_volume': 'mean',
        'plateau_pressure': 'mean'
    })

    # Order as in SQL
    aggregate = aggregate.sort_values(by=['stay_id', 'charttime'])
    aggregate.to_csv(f"{output_path}/vent_parameters.csv", index=False)
    print("Ventilator parameters saved to:", f"{output_path}/vent_parameters.csv")
    return aggregate


In [ ]:
vent_parameters = create_vent_parameters(mimic_path, output_path)
print(vent_parameters.head())

Loading chartevents table...
Filtered Data Created
Ventilator parameters saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/vent_parameters.csv
    stay_id  subject_id   hadm_id           charttime  PEEP  tidal_volume  \
0  30000153    12466550  23998182 2174-09-29 12:00:00   5.0           NaN   
1  30000153    12466550  23998182 2174-09-29 12:01:00   5.0         496.0   
2  30000153    12466550  23998182 2174-09-29 15:25:00   5.0         537.5   
3  30000153    12466550  23998182 2174-09-29 17:10:00   5.0         626.0   
4  30000213    13180007  27543152 2162-06-21 05:00:00   5.0           NaN   

   plateau_pressure  
0              13.0  
1               NaN  
2              16.0  
3               NaN  
4               NaN  


In [ ]:
vent_parameters.describe()

,stay_id,subject_id,hadm_id,charttime,PEEP,tidal_volume,plateau_pressure
count,9.790450e+05,9.790450e+05,9.790450e+05,979045,8.780150e+05,899221.000000,2.921840e+05
mean,3.495258e+07,1.501995e+07,2.502485e+07,2153-09-01 07:35:26.487833600,2.187163e+01,465.177898,7.431331e+02
min,3.000015e+07,1.000098e+07,2.000015e+07,2110-01-11 13:00:00,-9.500000e+00,0.000000,0.000000e+00
25%,3.245285e+07,1.251347e+07,2.255594e+07,2134-02-25 09:00:00,5.000000e+00,369.500000,1.600000e+01
50%,3.499380e+07,1.506507e+07,2.505412e+07,2153-10-05 16:44:00,5.000000e+00,441.000000,2.000000e+01
75%,3.743515e+07,1.750715e+07,2.745620e+07,2173-05-02 16:00:00,8.450000e+00,509.500000,2.400000e+01
max,3.999955e+07,1.999999e+07,2.999962e+07,2214-05-06 07:00:00,8.774580e+06,820914.000000,2.111110e+08
std,2.888985e+06,2.880452e+06,2.857995e+06,NaN,1.016928e+04,1976.321569,3.905554e+05


In [ ]:
import pandas as pd

def create_ventilation_params(mimic_path, output_path):
    """
    Extracts FiO2 and mechanical ventilation status per charttime, following getVentilationparams2.sql logic.

    Parameters:
        mimic_path (str): Path to MIMIC-III chartevents data (.csv or .csv.gz)
        output_path (str): Path to save the output DataFrame as CSV

    Returns:
        pd.DataFrame: DataFrame of grouped ventilation parameters
    """
    print("Loading chartevents table...")
    ce = pd.read_csv(f"{mimic_path}/icu/chartevents.csv.gz",
                     usecols=['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valuenum'],
                     parse_dates=['charttime'])

    FIO2_itemids = [223835, 3420, 3422, 190]
    MechVent_itemids = [720, 223848, 223849, 467, 445, 448, 449, 450, 1340, 1486, 1600, 224687, 639, 654, 681, 682, 683, 684, 224685, 224684, 224686, 218, 436, 535, 444, 459, 224697, 224695, 224696, 224746, 224747, 221, 1, 1211, 1655, 2000, 226873, 224738, 224419, 224750, 227187, 543, 5865, 5866, 224707, 224709, 224705, 224706, 60, 437, 505, 506, 686, 220339, 224700, 3459, 501, 502, 503, 224702, 223, 667, 668, 669, 670, 671, 672, 224701]

    # Filter for required itemids and non-null value
    filt = ce['value'].notnull()  & (
        ce['itemid'].isin(FIO2_itemids + MechVent_itemids)
    )
    ce_filt = ce.loc[filt, [
        'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valuenum'
    ]].copy()

    print("Filtered Data Created")

    # FIO2 normalization
    def fio2_normalized(row):
        iid = row['itemid']
        vn = row['valuenum']
        if iid == 223835:
            if pd.notnull(vn):
                if 0 < vn <= 1:
                    return vn * 100
                elif 1 < vn < 21:
                    return None
                elif 21 <= vn <= 100:
                    return vn
        elif iid in [3420, 3422]:
            return vn
        elif iid == 190:
            if pd.notnull(vn) and 0.20 < vn < 1:
                return vn * 100
        return None

    ce_filt['fio2_chartevents'] = ce_filt.apply(fio2_normalized, axis=1)

    # Mechanical ventilation flag, closely following case conditions
    def mech_vent_flag(row):
        iid = row['itemid']
        value = str(row['value'])
        if pd.isnull(iid) or pd.isnull(row['value']):
            return 0
        if iid == 720 and value != 'Other/Remarks':
            return 1
        if iid == 223848 and value != 'Other':
            return 1
        if iid == 223849:
            return 1
        if iid == 467 and value == 'Ventilator':
            return 1
        if iid in [445, 448, 449, 450, 1340, 1486, 1600, 224687, 639, 654, 681, 682, 683, 684, 224685, 224684, 224686, 218, 436, 535, 444, 459, 224697, 224695, 224696, 224746, 224747, 221, 1, 1211, 1655, 2000, 226873, 224738, 224419, 224750, 227187, 543, 5865, 5866, 224707, 224709, 224705, 224706, 60, 437, 505, 506, 686, 220339, 224700, 3459, 501, 502, 503, 224702, 223, 667, 668, 669, 670, 671, 672, 224701]:
            return 1
        return 0

    ce_filt['MechVent'] = ce_filt.apply(mech_vent_flag, axis=1)

    # Grouping: get max across all records per group (SQL case logic guarantees max is used for both)
    group_cols = ['subject_id', 'hadm_id', 'stay_id', 'charttime']
    agg = ce_filt.groupby(group_cols, as_index=False).agg({
        'fio2_chartevents': 'max',
        'MechVent': 'max'
    })

    agg.to_csv(f"{output_path}/ventilation_params.csv", index=False)
    print("Ventilation parameters saved to:", f"{output_path}/ventilation_params.csv")
    return agg


In [ ]:
ventilation_params = create_ventilation_params(mimic_path, output_path)
print(ventilation_params.head())

Loading chartevents table...
Filtered Data Created
Ventilation parameters saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/ventilation_params.csv
   subject_id   hadm_id   stay_id           charttime  fio2_chartevents  \
0    10000690  25860671  37081114 2150-11-02 20:27:00              70.0   
1    10000690  25860671  37081114 2150-11-03 00:50:00              50.0   
2    10000690  25860671  37081114 2150-11-03 04:38:00             100.0   
3    10000690  25860671  37081114 2150-11-03 08:00:00             100.0   
4    10000690  25860671  37081114 2150-11-03 09:00:00              70.0   

   MechVent  
0         0  
1         0  
2         0  
3         0  
4         0  


In [ ]:
import pandas as pd

def create_vasopressors(vaso_path, output_path):
    """
    Recreates the getVasopressors2 SQL logic in Python using CSV files for each vasopressor dose.

    Parameters:
        mimic_path (str): Path where CSV files for each vasopressor dose are stored.
            Expected files: norepinephrine_dose.csv, epinephrine_dose.csv,
                            phenylephrine_dose.csv, dopamine_dose.csv, vasopressin_dose.csv
        output_path (str): Path to save the merged vasopressors output CSV.

    Returns:
        pd.DataFrame: DataFrame with combined vasopressor rates and total vaso calculation.
    """
    print("Loading vasopressors tables...")
    # Load all vaso CSV files
    norepi = pd.read_csv(f"{vaso_path}/norepinephrine_dose.csv", usecols=['icustay_id', 'starttime', 'vaso_rate'])
    epi = pd.read_csv(f"{vaso_path}/epinephrine_dose.csv", usecols=['icustay_id', 'starttime', 'vaso_rate'])
    phenylep = pd.read_csv(f"{vaso_path}/phenylephrine_dose.csv", usecols=['icustay_id', 'starttime', 'vaso_rate'])
    dopamine = pd.read_csv(f"{vaso_path}/dopamine_dose.csv", usecols=['icustay_id', 'starttime', 'vaso_rate'])
    vasopressin = pd.read_csv(f"{vaso_path}/vasopressin_dose.csv", usecols=['icustay_id', 'starttime', 'vaso_rate'])

    print("Data Loading Completed")

    # Add columns for each vasopressor with nulls except the relevant dose
    norepi['rate_norepinephrine'] = norepi['vaso_rate']
    norepi = norepi.drop(columns=['vaso_rate'])
    norepi['rate_epinephrine'] = None
    norepi['rate_phenylephrine'] = None
    norepi['rate_dopamine'] = None
    norepi['rate_vasopressin'] = None

    epi['rate_norepinephrine'] = None
    epi['rate_epinephrine'] = epi['vaso_rate']
    epi['rate_phenylephrine'] = None
    epi['rate_dopamine'] = None
    epi['rate_vasopressin'] = None
    epi = epi.drop(columns=['vaso_rate'])

    phenylep['rate_norepinephrine'] = None
    phenylep['rate_epinephrine'] = None
    phenylep['rate_phenylephrine'] = phenylep['vaso_rate']
    phenylep['rate_dopamine'] = None
    phenylep['rate_vasopressin'] = None
    phenylep = phenylep.drop(columns=['vaso_rate'])
    print("Filtered Data Created")
    dopamine['rate_norepinephrine'] = None
    dopamine['rate_epinephrine'] = None
    dopamine['rate_phenylephrine'] = None
    dopamine['rate_dopamine'] = dopamine['vaso_rate']
    dopamine['rate_vasopressin'] = None
    dopamine = dopamine.drop(columns=['vaso_rate'])

    vasopressin['rate_norepinephrine'] = None
    vasopressin['rate_epinephrine'] = None
    vasopressin['rate_phenylephrine'] = None
    vasopressin['rate_dopamine'] = None
    vasopressin['rate_vasopressin'] = vasopressin['vaso_rate']
    vasopressin = vasopressin.drop(columns=['vaso_rate'])

    # Concatenate all rows (union all)
    union_df = pd.concat([norepi, epi, phenylep, dopamine, vasopressin], ignore_index=True)
    print("Union Data Created")
    # Convert all rate columns to numeric (in case some None are objects)
    rate_cols = ['rate_norepinephrine', 'rate_epinephrine', 'rate_phenylephrine', 'rate_dopamine', 'rate_vasopressin']
    for col in rate_cols:
        union_df[col] = pd.to_numeric(union_df[col], errors='coerce')

    # Group by icustay_id and starttime, taking max of each rate as in SQL
    vaso = union_df.groupby(['icustay_id', 'starttime'], as_index=False)[rate_cols].max()

    # Calculate vaso_total as per SQL formula
    vaso['vaso_total'] = (
        vaso['rate_norepinephrine'].fillna(0) +
        vaso['rate_epinephrine'].fillna(0) +
        vaso['rate_phenylephrine'].fillna(0) / 2.2 +
        vaso['rate_dopamine'].fillna(0) / 100 +
        vaso['rate_vasopressin'].fillna(0) * 8.33
    )

    # Sort by icustay_id, starttime
    vaso = vaso.sort_values(['icustay_id', 'starttime']).reset_index(drop=True)

    # Save to CSV
    vaso.to_csv(f"{output_path}/vasopressors_combined.csv", index=False)
    print("Vasopressors combined data saved to:", f"{output_path}/vasopressors_combined.csv")

    return vaso


In [ ]:
vasopressors = create_vasopressors(vaso_path, output_path)
print(vasopressors.head())

Loading vasopressors tables...
Data Loading Completed
Filtered Data Created


/tmp/ipython-input-3826096295.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  union_df = pd.concat([norepi, epi, phenylep, dopamine, vasopressin], ignore_index=True)


Union Data Created
Vasopressors combined data saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/vasopressors_combined.csv
   icustay_id            starttime  rate_norepinephrine  rate_epinephrine  \
0    30000484  2136-01-15 09:40:00                  NaN               NaN   
1    30000484  2136-01-15 17:42:00                  NaN               NaN   
2    30000484  2136-01-15 21:02:00                  NaN               NaN   
3    30000646  2194-04-29 08:54:00                  NaN               NaN   
4    30000646  2194-04-29 09:08:00                  NaN               NaN   

   rate_phenylephrine  rate_dopamine  rate_vasopressin  vaso_total  
0                 NaN       5.003784               NaN    0.050038  
1                 NaN       2.501892               NaN    0.025019  
2                 NaN       1.557223               NaN    0.015572  
3            0.500217            NaN               NaN    0.227371  
4            0.

In [ ]:
vasopressors.describe()

,icustay_id,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_dopamine,rate_vasopressin,vaso_total
count,7.475050e+05,459765.000000,31488.000000,209361.000000,18084.000000,37163.000000,747505.000000
mean,3.500916e+07,0.155127,0.154227,1.372656,7.954205,2.413129,1.277946
std,2.891239e+06,0.812412,0.488809,5.259600,43.043475,13.673560,25.800239
min,3.000048e+07,0.000200,0.000801,-5.724638,0.200002,0.016635,-2.602108
25%,3.250932e+07,0.050018,0.020148,0.500083,3.498957,1.703514,0.062322
50%,3.501621e+07,0.100029,0.050006,0.920834,5.020185,2.400000,0.153088
75%,3.751214e+07,0.199949,0.115171,1.622256,10.000250,2.404819,0.363798
max,3.999955e+07,359.550595,41.142862,880.058706,4000.000000,2400.000000,19992.000000


In [ ]:
import pandas as pd
import numpy as np

def create_vasopressors_charttable(mimic_path, output_path):
    """
    Replicates the SQL:
        SELECT ic.subject_id, ic.hadm_id, ic.icustay_id, starttime as charttime,
               rate_norepinephrine, rate_epinephrine, rate_phenylephrine,
               rate_vasopressin, rate_dopamine, vaso_total,
               + all other columns as NULL
        FROM getVasopressors2
        INNER JOIN icustays ic on ic.icustay_id = getVasopressors2.icustay_id

    Parameters:
        getvasopressors_path (str): Path to getVasopressors2 CSV
        icustays_path (str): Path to icustays CSV
        output_path (str): Optional. Save resulting DataFrame to CSV.

    Returns:
        pandas.DataFrame: DataFrame equivalent to SQL query output
    """

    print("Loading input data...")
    vaso = pd.read_csv(f"{output_path}/vasopressors_combined.csv", parse_dates=["starttime"])
    vaso = vaso.rename(columns={'starttime': 'charttime','icustay_id': 'stay_id'})
    # icu = pd.read_csv(icustays_path)
    icu = pd.read_csv(f"{mimic_path}/icu/icustays.csv.gz",
                     usecols=['subject_id', 'hadm_id', 'stay_id'])

    print("Joining vasopressor data with icustays...")
    df = vaso.merge(icu[['stay_id', 'subject_id', 'hadm_id']],
                    on='stay_id', how='inner')



    # Reorder columns`
    df = df[['subject_id', 'hadm_id', 'stay_id', 'charttime',
             'rate_norepinephrine', 'rate_epinephrine', 'rate_phenylephrine',
             'rate_vasopressin', 'rate_dopamine', 'vaso_total']]
     # Save to CSV
    df.to_csv(f"{output_path}/vasopressors_combined_final.csv", index=False)
    print("Vasopressors combined data saved to:", f"{output_path}/vasopressors_combined_final.csv")

    return df

In [ ]:
vasopressors_final = create_vasopressors_charttable(mimic_path, output_path)
print(vasopressors_final.head())
vasopressors_final.describe()

Loading input data...
Joining vasopressor data with icustays...
Vasopressors combined data saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/vasopressors_combined_final.csv
   subject_id   hadm_id   stay_id           charttime  rate_norepinephrine  \
0    18421337  22413411  30000484 2136-01-15 09:40:00                  NaN   
1    18421337  22413411  30000484 2136-01-15 17:42:00                  NaN   
2    18421337  22413411  30000484 2136-01-15 21:02:00                  NaN   
3    12207593  22795209  30000646 2194-04-29 08:54:00                  NaN   
4    12207593  22795209  30000646 2194-04-29 09:08:00                  NaN   

   rate_epinephrine  rate_phenylephrine  rate_vasopressin  rate_dopamine  \
0               NaN                 NaN               NaN       5.003784   
1               NaN                 NaN               NaN       2.501892   
2               NaN                 NaN               NaN       1.557223   

,subject_id,hadm_id,stay_id,charttime,rate_norepinephrine,rate_epinephrine,rate_phenylephrine,rate_vasopressin,rate_dopamine,vaso_total
count,7.475050e+05,7.475050e+05,7.475050e+05,747505,459765.000000,31488.000000,209361.000000,37163.000000,18084.000000,747505.000000
mean,1.497235e+07,2.501058e+07,3.500916e+07,2153-12-19 10:30:42.007505920,0.155127,0.154227,1.372656,2.413129,7.954205,1.277946
min,1.000069e+07,2.000009e+07,3.000048e+07,2110-01-13 12:00:00,0.000200,0.000801,-5.724638,0.016635,0.200002,-2.602108
25%,1.242677e+07,2.255761e+07,3.250932e+07,2133-10-28 18:35:00,0.050018,0.020148,0.500083,1.703514,3.498957,0.062322
50%,1.496389e+07,2.506245e+07,3.501621e+07,2154-01-08 21:24:00,0.100029,0.050006,0.920834,2.400000,5.020185,0.153088
75%,1.749082e+07,2.738590e+07,3.751214e+07,2174-01-06 21:27:00,0.199949,0.115171,1.622256,2.404819,10.000250,0.363798
max,1.999984e+07,2.999939e+07,3.999955e+07,2211-05-03 15:48:00,359.550595,41.142862,880.058706,2400.000000,4000.000000,19992.000000
std,2.908182e+06,2.838274e+06,2.891239e+06,NaN,0.812412,0.488809,5.259600,13.673560,43.043475,25.800239


In [ ]:
import pandas as pd

def create_others(mimic_path, output_path):
    """
    Converts the getOthers SQL materialized view logic into Python with Pandas.
    Extracts and averages SGOT, SGPT, and Ionized Calcium from chartevents.

    Parameters:
        mimic_path (str): Path to MIMIC-III CHARTEVENTS CSV file
        output_path (str): Path to save the resulting CSV with aggregated lab values

    Returns:
        pd.DataFrame: Aggregated dataframe with SGOT, SGPT, IonizedCalcium averages per icustay_id and charttime
    """
    print("Loading CHARTEVENTS...")
    ce = pd.read_csv(f"{mimic_path}/icu/chartevents.csv.gz",
                     usecols=['subject_id', 'hadm_id', 'stay_id', 'charttime', 'valuenum', 'itemid'],
                     parse_dates=['charttime'])

    # Define itemid groups per labs of interest
    SGOT_ids = [220587]
    SGPT_ids = [220644]
    IonizedCalcium_ids = [225667]

    # Filter chartevents for relevant itemids and exclude error=1
    filtered_ce = ce[ce['itemid'].isin(SGOT_ids + SGPT_ids + IonizedCalcium_ids)].copy()
    print("Filtered Data Created")

    # Create separate columns with valuenum applied only when itemid matches (else None)
    filtered_ce['SGOT'] = filtered_ce.apply(lambda row: row['valuenum'] if row['itemid'] in SGOT_ids else None, axis=1)
    print("SGOT Created")
    filtered_ce['SGPT'] = filtered_ce.apply(lambda row: row['valuenum'] if row['itemid'] in SGPT_ids else None, axis=1)
    print("SGPT Created")
    filtered_ce['IonizedCalcium'] = filtered_ce.apply(lambda row: row['valuenum'] if row['itemid'] in IonizedCalcium_ids else None, axis=1)
    print("IonizedCalcium Created")

    # Group by subject_id, hadm_id, icustay_id, charttime and average each lab
    grouped = filtered_ce.groupby(
        ['subject_id', 'hadm_id', 'stay_id', 'charttime'],
        as_index=False
    ).agg({
        'SGOT': 'mean',
        'SGPT': 'mean',
        'IonizedCalcium': 'mean'
    })

    # Save to CSV
    grouped.to_csv(f"{output_path}/others_lab_values.csv", index=False)
    print("Aggregated lab values saved to:", f"{output_path}/others_lab_values.csv")

    return grouped


In [ ]:
others = create_others(mimic_path, output_path)
print(others.head())

Loading CHARTEVENTS...
Filtered Data Created
SGOT Created
SGPT Created
IonizedCalcium Created
Aggregated lab values saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/others_lab_values.csv
   subject_id   hadm_id   stay_id           charttime   SGOT   SGPT  \
0    10001843  26133978  39698942 2134-12-05 18:11:00  140.0   78.0   
1    10001843  26133978  39698942 2134-12-06 03:29:00  196.0   90.0   
2    10001884  26184834  37510196 2131-01-11 06:31:00  134.0  167.0   
3    10001884  26184834  37510196 2131-01-11 06:37:00    NaN    NaN   
4    10001884  26184834  37510196 2131-01-12 03:34:00   55.0  116.0   

   IonizedCalcium  
0             NaN  
1             NaN  
2             NaN  
3            1.25  
4             NaN  


In [ ]:
others.describe()

,subject_id,hadm_id,stay_id,charttime,SGOT,SGPT,IonizedCalcium
count,4.564780e+05,4.564780e+05,4.564780e+05,456478,166522.000000,166575.000000,293569.000000
mean,1.499546e+07,2.499995e+07,3.500599e+07,2153-07-06 03:44:51.693445120,372.330488,283.071338,171.446937
min,1.000184e+07,2.000009e+07,3.000015e+07,2110-01-11 11:50:00,0.000000,0.000000,0.000000
25%,1.248942e+07,2.248938e+07,3.251734e+07,2133-05-23 03:28:44.999999488,27.000000,18.000000,1.070000
50%,1.499086e+07,2.503543e+07,3.502154e+07,2153-07-14 17:11:30,51.000000,37.000000,1.120000
75%,1.749978e+07,2.744165e+07,3.749897e+07,2173-08-16 17:50:44.999999488,124.000000,100.000000,1.180000
max,1.999999e+07,2.999983e+07,3.999986e+07,2214-07-26 01:20:00,999999.000000,999999.000000,999999.000000
std,2.888913e+06,2.870124e+06,2.884334e+06,NaN,8217.650752,8160.953603,13049.465802


In [ ]:
others.isna().sum()

,0
subject_id,0
hadm_id,0
stay_id,0
charttime,0
SGOT,289956
SGPT,289903
IonizedCalcium,162909


In [ ]:
import pandas as pd

def create_SIRS_sampled_withventparams(output_path):
    """
    Converts the SQL logic for getSIRS_sampled_withventparams into Python using Pandas.
    Calculates SIRS score based on temperature, heart rate, respiratory rate, PaCO2, WBC, and bands.

    Parameters:
        sampled_data_path (str): Path to the CSV file containing the sampled_all_withventparams dataframe.
        output_path (str): Path to save the resulting CSV with SIRS scores

    Returns:
        pd.DataFrame: DataFrame with SIRS scores
    """
    print(f"Loading data from {output_path}...")

    # Load the pre-combined data
    scorecomp = pd.read_csv(f"{output_path}/sampled_all_withventparams.csv",
                            parse_dates=['start_time'])

    print("Data loaded.")

    # --- scorecalc logic ---
    scorecalc = scorecomp.copy()

    scorecalc['Temp_score'] = scorecalc['TempC'].apply(
        lambda x: 1 if pd.notnull(x) and (x < 36.0 or x > 38.0) else (0 if pd.notnull(x) else None)
    )

    scorecalc['HeartRate_score'] = scorecalc['HeartRate'].apply(
        lambda x: 1 if pd.notnull(x) and x > 90.0 else (0 if pd.notnull(x) else None)
    )

    scorecalc['Resp_score'] = scorecalc.apply(
        lambda row: 1 if (pd.notnull(row['RespRate']) and row['RespRate'] > 20.0) or
                         (pd.notnull(row['PACO2']) and row['PACO2'] < 32.0)
                      else (0 if pd.notnull(row['RespRate']) or pd.notnull(row['PACO2']) else None),
        axis=1
    )

    scorecalc['WBC_score'] = scorecalc.apply(
        lambda row: 1 if (pd.notnull(row['WBC']) and (row['WBC'] < 4.0 or row['WBC'] > 12.0)) or
                         (pd.notnull(row['BANDS']) and row['BANDS'] > 10)
                      else (0 if pd.notnull(row['WBC']) or pd.notnull(row['BANDS']) else None),
        axis=1
    )

    print("Score calculations completed.")

    # --- Final SELECT and SIRS calculation ---
    final_df = scorecalc[['stay_id', 'subject_id', 'hadm_id', 'start_time',
                          'Temp_score', 'HeartRate_score', 'Resp_score', 'WBC_score']].copy()

    # Calculate SIRS score (impute 0 for missing scores)
    final_df['SIRS'] = final_df[['Temp_score', 'HeartRate_score', 'Resp_score', 'WBC_score']].fillna(0).sum(axis=1)

    # Reorder columns and select final output columns
    final_df = final_df[['stay_id', 'subject_id', 'hadm_id', 'start_time', 'SIRS',
                         'Temp_score', 'HeartRate_score', 'Resp_score', 'WBC_score']]

    # Save to CSV
    output_filepath = f"{output_path}/SIRS_sampled_withventparams.csv"
    final_df.to_csv(output_filepath, index=False)
    print("SIRS scores saved to:", output_path)

    return final_df

In [ ]:
SIRS_sampled_withventparams = create_SIRS_sampled_withventparams(output_path)
print(SIRS_sampled_withventparams.head())

Loading data from /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data...
Data loaded.
Score calculations completed.
SIRS scores saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data
    stay_id  subject_id     hadm_id          start_time  SIRS  Temp_score  \
0  30000153  12466550.0  23998182.0 2174-09-29 12:00:00   2.0         0.0   
1  30000153  12466550.0  23998182.0 2174-09-29 16:00:00   1.0         0.0   
2  30000153  12466550.0  23998182.0 2174-09-29 20:00:00   2.0         1.0   
3  30000153  12466550.0  23998182.0 2174-09-30 00:00:00   2.0         0.0   
4  30000153  12466550.0  23998182.0 2174-09-30 04:00:00   1.0         0.0   

   HeartRate_score  Resp_score  WBC_score  
0              1.0         0.0        1.0  
1              1.0         0.0        NaN  
2              1.0         0.0        NaN  
3              1.0         0.0        1.0  
4              1.0         0.0        NaN

In [ ]:
SIRS_sampled_withventparams.shape

(2248324, 9)

In [ ]:
SIRS_sampled_withventparams.isna().sum()

,0
stay_id,0
subject_id,0
hadm_id,0
start_time,0
SIRS,0
Temp_score,406093
HeartRate_score,149900
Resp_score,135217
WBC_score,1998013


In [1]:
import pandas as pd
import numpy as np

def create_SOFA_sampled_withventparams(output_path):
    """
    Converts the provided SQL logic for calculating SOFA scores into Python using Pandas.

    Parameters:
        data_path (str): Path to the CSV file containing the necessary columns
                         (icustay_id, subject_id, hadm_id, start_time, PLATELET, CREATININE, urineoutput,
                          PAO2FiO2ratio, MechVent, gcs, MeanBP, rate_dopamine, rate_norepinephrine,
                          rate_epinephrine, BILIRUBIN).
        output_path (str): Path to save the resulting CSV with SOFA scores.

    Returns:
        pd.DataFrame: DataFrame with SOFA scores and individual component scores.
    """
    print(f"Loading data from {output_path}...")
    # Load the data containing the necessary columns
    scorecomp = pd.read_csv(f"{output_path}/sampled_all_withventparams.csv", parse_dates=['start_time'])
    print("Data loaded.")

    scorecalc = scorecomp.copy()

    # --- Coagulation score ---
    # Use 'PLATELET' as requested
    conditions_coag = [
        scorecalc['PLATELET'] < 20,
        scorecalc['PLATELET'] < 50,
        scorecalc['PLATELET'] < 100,
        scorecalc['PLATELET'] < 150,
        scorecalc['PLATELET'].isnull()
    ]
    choices_coag = [4, 3, 2, 1, None]
    scorecalc['coagulation'] = np.select(conditions_coag, choices_coag, default=0)

    # --- Renal failure score ---
    conditions_renal = [
        (scorecalc['CREATININE'] >= 5.0) | (scorecalc['urineoutput'] < 200),
        (scorecalc['CREATININE'] >= 3.5) | (scorecalc['urineoutput'] < 500),
        (scorecalc['CREATININE'] >= 2.0),
        (scorecalc['CREATININE'] >= 1.2),
        scorecalc['urineoutput'].isnull() & scorecalc['CREATININE'].isnull()
    ]
    choices_renal = [4, 3, 2, 1, None]
    scorecalc['renal'] = np.select(conditions_renal, choices_renal, default=0)


    # --- Respiration score ---
    conditions_resp = [
        (pd.notnull(scorecalc['PAO2FiO2ratio'])) & (scorecalc['PAO2FiO2ratio'] < 100) & (scorecalc['MechVent'] == 1),
        (pd.notnull(scorecalc['PAO2FiO2ratio'])) & (scorecalc['PAO2FiO2ratio'] < 200) & (scorecalc['MechVent'] == 1),
        (pd.notnull(scorecalc['PAO2FiO2ratio'])) & (scorecalc['PAO2FiO2ratio'] < 300),
        (pd.notnull(scorecalc['PAO2FiO2ratio'])) & (scorecalc['PAO2FiO2ratio'] < 400),
        scorecalc['PAO2FiO2ratio'].isnull()
    ]
    choices_resp = [4, 3, 2, 1, None]
    scorecalc['respiration'] = np.select(conditions_resp, choices_resp, default=0)


    # --- Neurological failure (GCS) score ---
    conditions_cns = [
        (scorecalc['gcs'] >= 13) & (scorecalc['gcs'] <= 14),
        (scorecalc['gcs'] >= 10) & (scorecalc['gcs'] <= 12),
        (scorecalc['gcs'] >= 6) & (scorecalc['gcs'] <= 9),
        scorecalc['gcs'] < 6,
        scorecalc['gcs'].isnull()
    ]
    choices_cns = [1, 2, 3, 4, None]
    scorecalc['cns'] = np.select(conditions_cns, choices_cns, default=0)


    # --- Cardiovascular score ---
    conditions_cardio = [
        (scorecalc['rate_dopamine'] > 15) | (scorecalc['rate_epinephrine'] > 0.1) | (scorecalc['rate_norepinephrine'] > 0.1),
        (scorecalc['rate_dopamine'] > 5) | (scorecalc['rate_epinephrine'] <= 0.1) | (scorecalc['rate_norepinephrine'] <= 0.1),
        scorecalc['rate_dopamine'] <= 5, # Assuming dobutamine logic is not needed based on provided SQL
        scorecalc['MeanBP'] < 70,
        scorecalc['MeanBP'].isnull() & scorecalc['rate_dopamine'].isnull() & scorecalc['rate_epinephrine'].isnull() & scorecalc['rate_norepinephrine'].isnull()
    ]
    choices_cardio = [4, 3, 2, 1, None]
    scorecalc['cardiovascular'] = np.select(conditions_cardio, choices_cardio, default=0)


    # --- Liver score ---
    conditions_liver = [
        scorecalc['BILIRUBIN'] >= 12.0,
        scorecalc['BILIRUBIN'] >= 6.0,
        scorecalc['BILIRUBIN'] >= 2.0,
        scorecalc['BILIRUBIN'] >= 1.2,
        scorecalc['BILIRUBIN'].isnull()
    ]
    choices_liver = [4, 3, 2, 1, None]
    scorecalc['liver'] = np.select(conditions_liver, choices_liver, default=0)


    print("Individual component scores calculated.")

    # --- overall SOFA score calculation ---
    scorecalc['SOFA'] = scorecalc[['respiration', 'cns', 'cardiovascular', 'liver', 'coagulation', 'renal']].fillna(0).sum(axis=1)


    # Select the necessary columns: identifying columns, input columns for SOFA, and calculated scores
    final_columns = [
        'stay_id', 'subject_id', 'hadm_id', 'start_time',
        'PAO2FiO2ratio', 'MechVent', 'gcs', 'MeanBP', 'rate_dopamine',
        'rate_norepinephrine', 'rate_epinephrine', 'BILIRUBIN',
        'PLATELET', 'CREATININE', 'urineoutput',
        'respiration', 'cns', 'cardiovascular', 'liver',
        'coagulation', 'renal', 'SOFA' # Include individual scores and total SOFA
    ]

    # Ensure all required columns are in scorecalc before selecting
    for col in final_columns:
        if col not in scorecalc.columns:
             # If a column is missing, add it with None values
            scorecalc[col] = None


    final_df = scorecalc[final_columns].copy()


    # Sort by stay_id, subject_id, hadm_id, start_time (using 'stay_id' to match the output format)
    final_df = final_df.sort_values(by=['stay_id', 'subject_id', 'hadm_id', 'start_time']).reset_index(drop=True)

    # Save to CSV
    output_filepath = f"{output_path}/sofa_scores.csv"
    final_df.to_csv(output_filepath, index=False)
    print("SOFA scores saved to:", output_filepath)

    return final_df

In [4]:
SOFA_sampled_withventparams = create_SOFA_sampled_withventparams(output_path)
print(SOFA_sampled_withventparams.head())

Loading data from /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data...
Data loaded.
Individual component scores calculated.


/tmp/ipython-input-333375864.py:100: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scorecalc['SOFA'] = scorecalc[['respiration', 'cns', 'cardiovascular', 'liver', 'coagulation', 'renal']].fillna(0).sum(axis=1)


SOFA scores saved to: /content/drive/My Drive/mimic-iv/physionet.org/files/mimiciv/3.0/ventilation_data/patient_data/sofa_scores.csv
    stay_id  subject_id     hadm_id          start_time  PAO2FiO2ratio  \
0  30000153  12466550.0  23998182.0 2174-09-29 12:00:00            NaN   
1  30000153  12466550.0  23998182.0 2174-09-29 16:00:00            NaN   
2  30000153  12466550.0  23998182.0 2174-09-29 20:00:00            NaN   
3  30000153  12466550.0  23998182.0 2174-09-30 00:00:00            NaN   
4  30000153  12466550.0  23998182.0 2174-09-30 04:00:00            NaN   

   MechVent   gcs  MeanBP  rate_dopamine  rate_norepinephrine  ...  PLATELET  \
0         1   NaN   84.25            NaN                  NaN  ...     173.0   
1         1   9.0   79.00            NaN                  NaN  ...       NaN   
2         0  12.0   87.20            NaN                  NaN  ...       NaN   
3         0  14.0   82.75            NaN                  NaN  ...     162.0   
4         0  14.0   92

In [ ]:
SOFA_sampled_withventparams.shape

(2248324, 22)

In [ ]:
SOFA_sampled_withventparams.isna().sum()

,0
stay_id,0
subject_id,0
hadm_id,0
start_time,0
PAO2FiO2ratio,2208895
MechVent,0
gcs,896150
MeanBP,166908
rate_dopamine,2239293
rate_norepinephrine,2068774
